# 02. Infinite Well via Dirichlet Sine-Transform Split Evolution

The previous periodic-QFT treatment evolved a free particle on a ring and compared it with a hard-wall sine-basis reference. This corrected notebook uses a Dirichlet sine transform, the boundary-compatible Fourier/QFT-family transform for the infinite well. The numerical evolution and analytical reference now use the same hard-wall boundary condition.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from qftsplit.core import (
    build_infinite_well_circuit,
    build_periodic_resource_circuit,
    configure_matplotlib,
    dirichlet_midpoint_grid,
    draw_and_save_circuit,
    fft_momentum_grid,
    fidelity_summary,
    harmonic_potential,
    periodic_grid,
    plot_convergence,
    plot_density_snapshots,
    plot_fidelity_vs_gate_count,
    plot_fidelity,
    qft_split_circuit_validation,
    run_harmonic_case,
    run_infinite_well_case,
    save_dataframe,
    save_publication_figure,
    sine_mode_energies,
    sine_transform_validation,
    transpile_and_extract_metrics,
)

FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
configure_matplotlib()

In [ ]:
L = 10.0
N = 64
hbar = 1.0
m = 1.0

x0 = L / 2.0
k0 = 2.0
sigma = 0.8

t_max = 6.0
r = 700
snapshot_count = 6

reference_grid_size = 4097
reference_tail_tolerance = 1e-10
reference_basis_cap_min = 192

fidelity_sweep_r = [100, 200, 300, 400, 500, 600, 700]
spatial_convergence_N = [32, 64, 128]

In [ ]:
result = run_infinite_well_case(
    grid_size=N,
    length=L,
    x0=x0,
    k0=k0,
    sigma=sigma,
    t_max=t_max,
    steps=r,
    mass=m,
    hbar=hbar,
    reference_grid_size=reference_grid_size,
    reference_tail_tolerance=reference_tail_tolerance,
    reference_basis_cap=max(reference_basis_cap_min, 4 * N),
)
summary = fidelity_summary(result)

display(Markdown("## Main Run Summary"))
display(pd.DataFrame([summary]))
display(
    Markdown(
        f"Reference basis kept **{result['reference_data']['n_keep']}** sine states out of cap "
        f"**{result['reference_data']['basis_cap']}**, with discarded tail estimate "
        f"**{result['reference_data']['tail_weight']:.2e}**."
    )
)

In [ ]:
parameter_df = pd.DataFrame(
    [
        {
            "system": "infinite_well",
            "numerical_transform": "orthonormal_DST-II_quantum_sine_transform_family",
            "boundary_model": "Dirichlet_hard_wall",
            "periodic_boundary_mismatch": False,
            "N": N,
            "n_qubits": int(np.log2(N)),
            "domain": f"(0, {L}) midpoint grid",
            "L": L,
            "x0": x0,
            "k0": k0,
            "sigma": sigma,
            "t_max": t_max,
            "r": r,
            "dt": result["dt"],
            "hbar": hbar,
            "m": m,
            "reference_grid_size": result["reference_data"]["dense_grid_size"],
            "reference_basis_cap": result["reference_data"]["basis_cap"],
            "reference_basis_kept": result["reference_data"]["n_keep"],
            "reference_tail_weight": result["reference_data"]["tail_weight"],
            "initial_state_modifier": "sin(pi x / L)",
            "boundary_caveat": "resolved_by_Dirichlet_sine_transform_not_periodic_FFT",
            "fidelity_sweep_r": ",".join(str(value) for value in fidelity_sweep_r),
            "spatial_convergence_N": ",".join(str(value) for value in spatial_convergence_N),
        }
    ]
)
fidelity_df = pd.DataFrame(
    {
        "time": result["times"],
        "fidelity": result["fidelity"],
        "split_norm": result["split_norms"],
        "reference_norm": result["reference_norms"],
    }
)

save_dataframe(parameter_df, TABLES_DIR, "infinite_well_parameters.csv")
save_dataframe(fidelity_df, TABLES_DIR, "infinite_well_fidelity_vs_time.csv")

display(Markdown("## Simulation Parameters"))
display(parameter_df)

In [ ]:
snapshot_figure = plot_density_snapshots(
    x=result["x"],
    times=result["times"],
    reference_states=result["reference_states"],
    split_states=result["split_states"],
    snapshot_count=snapshot_count,
    title="Infinite-well probability density",
)
fidelity_figure = plot_fidelity(result["times"], result["fidelity"], "Infinite-well fidelity")

for stale_path in list(FIGURES_DIR.glob("infinite_well_density_snapshots_page_*.png")) + list(FIGURES_DIR.glob("infinite_well_density_snapshots_page_*.pdf")):
    stale_path.unlink()

save_publication_figure(snapshot_figure, FIGURES_DIR, "infinite_well_density_snapshots")
save_publication_figure(fidelity_figure, FIGURES_DIR, "infinite_well_fidelity_vs_time")
plt.close(snapshot_figure)
plt.close(fidelity_figure)

print("Saved infinite-well density and fidelity figures.")

In [ ]:
sweep_records = []
for r_value in fidelity_sweep_r:
    local = run_infinite_well_case(
        grid_size=N,
        length=L,
        x0=x0,
        k0=k0,
        sigma=sigma,
        t_max=t_max,
        steps=int(r_value),
        mass=m,
        hbar=hbar,
        reference_grid_size=reference_grid_size,
        reference_tail_tolerance=reference_tail_tolerance,
        reference_basis_cap=max(reference_basis_cap_min, 4 * N),
    )
    local_summary = fidelity_summary(local)
    sweep_records.append(
        {
            "system": "infinite_well",
            "r": int(r_value),
            "dt": local["dt"],
            "final_time_fidelity": local_summary["final_time_fidelity"],
            "minimum_fidelity": local_summary["minimum_fidelity"],
            "mean_fidelity": local_summary["mean_fidelity"],
        }
    )

fidelity_sweep_df = pd.DataFrame(sweep_records)
save_dataframe(fidelity_sweep_df, TABLES_DIR, "infinite_well_fidelity_vs_r.csv")
display(Markdown("## Trotter-Step Convergence"))
display(fidelity_sweep_df)

In [ ]:
convergence_records = []
for n_value in spatial_convergence_N:
    local = run_infinite_well_case(
        grid_size=int(n_value),
        length=L,
        x0=x0,
        k0=k0,
        sigma=sigma,
        t_max=t_max,
        steps=r,
        mass=m,
        hbar=hbar,
        reference_grid_size=reference_grid_size,
        reference_tail_tolerance=reference_tail_tolerance,
        reference_basis_cap=max(reference_basis_cap_min, 4 * int(n_value)),
    )
    local_summary = fidelity_summary(local)
    convergence_records.append(
        {
            "system": "infinite_well",
            "N": int(n_value),
            "n_qubits": int(np.log2(n_value)),
            "dx": local["dx"],
            "final_time_fidelity": local_summary["final_time_fidelity"],
            "minimum_fidelity": local_summary["minimum_fidelity"],
        }
    )

spatial_convergence_df = pd.DataFrame(convergence_records)
save_dataframe(spatial_convergence_df, TABLES_DIR, "infinite_well_spatial_convergence.csv")
convergence_figure = plot_convergence(
    spatial_convergence_df,
    x_column="N",
    y_column="final_time_fidelity",
    title="Infinite-well spatial convergence",
    xlabel="Grid points, N",
)
save_publication_figure(convergence_figure, FIGURES_DIR, "infinite_well_spatial_convergence")
plt.close(convergence_figure)

display(Markdown("## Spatial Convergence"))
display(spatial_convergence_df)